In [1]:
import os

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
from peft import get_peft_model, LoraConfig, TaskType

/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from plm_compare_progen2 import *
from plm_compare_esm import *
from protein_data import *
from pro_gen2_lora import *

In [24]:
device = 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


In [ ]:
# filename = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/dict_domainome_uniprot.pkl'
# with open(filename, "rb") as f:
#     dict_uniprot = pickle.load(f)

In [18]:
filename = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/dict_domainome_uniprot_new.pkl'
with open(filename, "rb") as f:
    dict_uniprot = pickle.load(f)

In [9]:
load_dotenv()
cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

client = storage.Client()
bucket = client.bucket('domainome-data')
blob = bucket.blob('SupplementaryTable2.txt')

df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

In [5]:
len(dict_uniprot.keys())

428

In [17]:
def make_mutation_fitness_df(domain_id, df):

    domain_id_list=domain_id.split("_")
    dom_pos = float(domain_id_list[-1])

    df_one_protein = df.where(df['domain_ID'] == domain_id).dropna()

    dom_position = df_one_protein['position'] - dom_pos
    df_one_protein.insert(loc=0, column='real_position', value=dom_position)
    df_one_protein_ns = df_one_protein[df_one_protein['mut_aa'] != '*'].copy()

    df_one_protein_ns['wt_seq'] = df_one_protein_ns.apply(lambda row: 
                                                      insert_wt(row['aa_seq'], row['real_position'], row['wt_aa']),
                                                      axis=1)
    
    df_mutation = df_one_protein_ns[['wt_seq','real_position','mut_aa','normalized_fitness']]

    return df_mutation
    

In [16]:
df.head()

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,True,118.0,113.0,62.0,10.0,29.0,2.0,97.66667,0.030945,0.014885,-0.819050,0.208478,339
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,A,False,219.0,277.0,217.0,86.0,225.0,137.0,237.66670,0.069376,0.006673,-0.280790,0.093461,339
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,C,False,706.0,726.0,459.0,768.0,507.0,616.0,630.33330,0.082052,0.004141,-0.103250,0.057995,339
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,D,False,407.0,431.0,323.0,508.0,159.0,111.0,387.00000,0.071003,0.005162,-0.258003,0.072296,339
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,E,False,37.0,56.0,37.0,201.0,102.0,95.0,43.33333,0.116326,0.012085,0.376783,0.169263,339


In [19]:
dom_id = 'A0A2R8Y422_PF00240_2'

df_mutation = make_mutation_fitness_df(dom_id, df)

In [21]:
df_mutation.head()

,wt_seq,real_position,mut_aa,normalized_fitness
0,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,0.0,A,-0.280790
1,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,0.0,C,-0.103250
2,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,0.0,D,-0.258003
3,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,0.0,E,0.376783
4,QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,0.0,F,0.149047


In [23]:
protein_seq = df_mutation['wt_seq'].iloc[0]
protein_seq

'QIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVL'

In [61]:
dict_dn_fitness = {}

In [62]:
domain_ids = list(df['domain_ID'].unique())    

for dom_id in domain_ids:

    dict_dn_fitness[dom_id] = {}

    df_mutation = make_mutation_fitness_df(dom_id, df)
    protein_seq = df_mutation['wt_seq'].iloc[0]

    fitness_data = df_mutation
    fitness_data.reset_index(drop=True, inplace=True)
    positions = np.arange(len(protein_seq))

    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
    positions_col = np.repeat(positions, len(amino_acids))
    amino_acids_col = np.tile(list(amino_acids), len(positions))
    df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
    df_merged = df2.merge(
        fitness_data,
        on=['real_position', 'mut_aa'],
        how='left')
    fitness_list = df_merged['normalized_fitness'].tolist()

    dom_data = dom_id.split("_")

    dict_dn_fitness[dom_id]['dom_seq'] = protein_seq
    dict_dn_fitness[dom_id]['uniprot_id'] = dom_data[0]
    dict_dn_fitness[dom_id]['pfam'] = dom_data[1]
    dict_dn_fitness[dom_id]['domain_start'] = dom_data[2]
    dict_dn_fitness[dom_id]['fitness'] = fitness_list




save a dictionary as pickle

In [ ]:
# path = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/'
# with open(path+"dict_dn_fitness.pkl", "wb") as f:
#     pickle.dump(dict_dn_fitness, f)

In [65]:
keys = list(dict_dn_fitness.keys())
len(keys)

522

In [64]:
i = 1
key = keys[i]

print(key)

dict_dn_fitness[key]

A0PJY2_PF00096_289


{'dom_seq': 'VCKVCGKGFRQASTLCRHKIIH',
 'uniprot_id': 'A0PJY2',
 'pfam': 'PF00096',
 'domain_start': '289',
 'fitness': [-0.252455221983102,
  -0.801102574773856,
  0.213684880185916,
  -0.170356053945728,
  -0.314743717241087,
  -0.116247484145821,
  -0.202370375840998,
  0.215030502197182,
  -0.413446276091886,
  -0.28032860376098,
  0.179151315068184,
  0.0545966957823591,
  -0.779433995993926,
  -0.496912613469845,
  -0.422941571616979,
  -0.588839353594538,
  -0.250398805949204,
  nan,
  -0.730083706100204,
  -0.119415307689338,
  -1.08948148335698,
  nan,
  -0.104814316848709,
  -0.0806045952480325,
  -0.891953096427874,
  -0.901045798421561,
  -1.21533564949891,
  -1.06967763638766,
  -0.209464941891363,
  -0.465358096198127,
  -0.916563810678981,
  -0.571248990216425,
  -0.88377958502429,
  -0.491142916082251,
  -1.00329387443099,
  -0.975046321757678,
  -0.353037192888365,
  -0.728121934752032,
  -0.586921989217069,
  -0.993158876181781,
  -0.159890352852709,
  -0.1936168041045